# Skill 35: Transformer Encoder Block

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Assemble self-attention, residual connections, LayerNorm and a GELU feed-forward network into one block.

## Objectives
- Combine MHA with a position-wise feed-forward network.
- Use residual paths and normalization to stabilize training.
- Keep the input and output dimensions consistent.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$x'=x+\mathrm{MHA}(\mathrm{LN}(x)),\qquad y=x'+\mathrm{FFN}(\mathrm{LN}(x'))$$
$\mathrm{FFN}(u)=W_2\,\mathrm{GELU}(W_1u+b_1)+b_2$ with hidden size $4E$. Parameters per block: $12E^2+13E$ (49,984 for E=64).

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import torch.nn as nn

class EncoderBlock(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, mlp_ratio * embed_dim),
            nn.GELU(),
            nn.Linear(mlp_ratio * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        z = self.norm1(x)
        a, _ = self.attn(z, z, z, key_padding_mask=key_padding_mask,
                         need_weights=False)
        x = x + a
        x = x + self.ffn(self.norm2(x))
        return x

x = torch.randn(2, 10, 64)
block = EncoderBlock()
y = block(x)
print(y.shape)  # (2,10,64)


## What to check
**Expected result:** Input and output shapes match: `(2, 10, 64)`.

## Pitfalls
Use `batch_first=True`. Call `block.eval()` before comparing outputs (dropout). In `key_padding_mask`, `True` means ignore.